In [1]:
!pip install anthropic scikit-learn

import json
import random
import anthropic
from collections import defaultdict
from google.colab import userdata
from sklearn.metrics import cohen_kappa_score

In [6]:
!sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids
0 upgraded, 5 newly installed, 0 to remove and 53 not upgraded.
Need to get 884 kB of archives.
After this operation, 3,256 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1 [322 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/main amd64 usb.ids all 2022.04.02-1 [219 kB]


In [37]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(5)
print("Ollama 서버 실행 완료")

Ollama 서버 실행 완료


In [38]:
!ollama pull qwen2.5:7b

In [39]:
from google.colab import files
import json
import random
import requests
from collections import defaultdict
from sklearn.metrics import cohen_kappa_score

uploaded = files.upload()

DATA_PATH   = "contract_risk_augmented_v1.json"
OUTPUT_PATH = "iaa_result.json"
SAMPLE_N    = 2

with open(DATA_PATH, encoding="utf-8") as f:
    data = json.load(f)

Saving contract_risk_augmented_v1.json to contract_risk_augmented_v1 (7).json


In [40]:
# 샘플 추출
sample_map = defaultdict(list)
for item in data:
    lines     = item["output"].split("\n")
    verdict   = lines[0].replace("판정: ", "").strip()
    type_name = lines[1].replace("유형: ", "").strip()
    sample_map[(type_name, verdict)].append(item)

samples = []
for key, items in sample_map.items():
    samples.extend(random.sample(items, min(SAMPLE_N, len(items))))

print(f"샘플 수: {len(samples)}개")

샘플 수: 58개


In [41]:
SYSTEM_PROMPT = """당신은 공공기관 SW 용역계약 조항 분석 전문가입니다.
계약 조항을 읽고 아래 형식으로만 답하세요.

판정: (정상/누락/위반 중 하나)
유형: (위험 유형명)
근거: (판단 근거)"""

def evaluate_with_qwen(item):
    input_text = item["input"]
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "qwen2.5:7b",
                "prompt": f"{SYSTEM_PROMPT}\n\n{input_text}",
                "stream": False
            }
        )
        return response.json()["response"].strip()
    except Exception as e:
        print(f"오류: {e}")
        return None

import time
for i, item in enumerate(samples):
    item["qwen_output"] = evaluate_with_qwen(item)
    print(f"{i+1}/{len(samples)} 완료")
    time.sleep(1)

print("Qwen 판정 완료")

1/58 완료
2/58 완료
3/58 완료
4/58 완료
5/58 완료
6/58 완료
7/58 완료
8/58 완료
9/58 완료
10/58 완료
11/58 완료
12/58 완료
13/58 완료
14/58 완료
15/58 완료
16/58 완료
17/58 완료
18/58 완료
19/58 완료
20/58 완료
21/58 완료
22/58 완료
23/58 완료
24/58 완료
25/58 완료
26/58 완료
27/58 완료
28/58 완료
29/58 완료
30/58 완료
31/58 완료
32/58 완료
33/58 완료


KeyboardInterrupt: 

In [24]:
def extract_verdict(output):
    if not output:
        return None
    for line in output.split("\n"):
        if line.startswith("판정:"):
            return line.replace("판정:", "").strip()
    return None

gpt_verdicts  = []
qwen_verdicts = []
valid_samples = []

for item in samples:
    gpt_v  = extract_verdict(item["output"])
    qwen_v = extract_verdict(item["qwen_output"])

    if gpt_v and qwen_v:
        gpt_verdicts.append(gpt_v)
        qwen_verdicts.append(qwen_v)
        valid_samples.append(item)

matches = sum(g == q for g, q in zip(gpt_verdicts, qwen_verdicts))
print(f"유효 샘플: {len(valid_samples)}개")
print(f"일치: {matches}/{len(valid_samples)} ({matches/len(valid_samples)*100:.1f}%)")

유효 샘플: 58개
일치: 38/58 (65.5%)


In [25]:
# Cohen's Kappa 계산
kappa = cohen_kappa_score(gpt_verdicts, qwen_verdicts)
print(f"Cohen's Kappa: {kappa:.4f}")

if kappa >= 0.8:
    print("우수")
elif kappa >= 0.6:
    print("양호")
elif kappa >= 0.4:
    print("보통 (일부 재검토 필요)")
else:
    print("미흡 (데이터 재검토 필요)")

Cohen's Kappa: 0.4817
보통 (일부 재검토 필요)


In [14]:
# 불일치 케이스 확인
mismatches = [
    {
        "계약조항": item["input"].split("\n")[1],
        "GPT판정":  gpt_v,
        "Qwen판정": qwen_v,
        "GPT근거":  item["output"],
        "Qwen근거": item["qwen_output"],
    }
    for item, gpt_v, qwen_v in zip(valid_samples, gpt_verdicts, qwen_verdicts)
    if gpt_v != qwen_v
]

print(f"불일치 케이스: {len(mismatches)}개")
for i, m in enumerate(mismatches):
    print(f"\n[{i+1}] {m['계약조항'][:50]}")
    print(f"  GPT: {m['GPT판정']} / Qwen: {m['Qwen판정']}")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump({"kappa": kappa, "mismatches": mismatches}, f, ensure_ascii=False, indent=2)

from google.colab import files
files.download(OUTPUT_PATH)
print("저장 완료")

불일치 케이스: 19개

[1] 계약 이행 지연 시, 을은 지연에 대한 배상 의무를 진다.
  GPT: 누락 / Qwen: 정상

[2] 계약 이행 지연 시, 을은 지연으로 인한 벌금을 납부해야 한다.
  GPT: 누락 / Qwen: 정상

[3] 을은 계약 이행 지연 시 지연배상금을 지급하여야 하며, 이는 계약에 명시된 방식에 따른다.
  GPT: 누락 / Qwen: 정상

[4] 계약 이행이 지연될 경우 을은 지체상금을 납부해야 하며, 이는 계약 체결 시 합의된 기준에
  GPT: 누락 / Qwen: 정상

[5] 용역이 계약 기한 내 완료되지 않을 경우, 을은 지체일수에 따라 계약 금액의 0.35%를 
  GPT: 위반 / Qwen: 정상

[6] 을이 대금 청구서를 제출하면, 갑은 검토 후 대금을 지급한다.
  GPT: 누락 / Qwen: 위반

[7] 갑은 을로부터 대금 청구서를 수령한 후 대금을 지급한다.
  GPT: 누락 / Qwen: 위반

[8] 갑은 을에게 과업 수행 중 발생하는 모든 변경 사항을 설명할 수 있다.
  GPT: 누락 / Qwen: 위반

[9] 과업 변경에 따른 추가 비용은 을이 부담한다.
  GPT: 누락 / Qwen: 정상

[10] 갑은 필요한 경우 을에게 추가 과업을 요청할 수 있으며 을은 성실히 수행하여야 한다.
  GPT: 누락 / Qwen: 정상

[11] 을은 계약이행 중 발생하는 사유에 따라 갑의 승인을 받아 과업내용을 조정할 수 있으며, 이
  GPT: 누락 / Qwen: 정상

[12] 을은 갑의 요청에 따라 과업을 조정할 수 있으며, 필요한 경우 세부사항을 수정할 수 있다.
  GPT: 누락 / Qwen: 정상

[13] 을은 계약금액이 감소되었을 경우 본 계약을 해지할 수 있다. 단, 용역수행 정지기간에 대한
  GPT: 누락 / Qwen: 위반

[14] 을은 계약내용의 변경으로 계약금액이 100분의 40 이상 감소되었거나, 용역수행정지기간이 
  GPT: 누락 / Qwen: 정상

[15] 해당 계약

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

저장 완료


In [15]:
import json

with open("contract_risk_augmented_v1.json", encoding="utf-8") as f:
    data = json.load(f)

# 제거할 계약조항 키워드
remove_keywords = [
    "검토 후 대금을 지급한다",                          # 6번
    "대금 청구서를 수령한 후 대금을 지급한다",            # 7번
    "발생하는 모든 변경 사항을 설명할 수 있다",           # 8번
    "계약금액이 감소되었을 경우 본 계약을 해지할 수 있다", # 13번
    "지식재산권은 갑과 을이 공동으로 소유한다",           # 15번 (IP 유형)
    "발주자의 재량에 따른다",                            # 19번
]

before = len(data)
filtered = [
    item for item in data
    if not any(kw in item["input"] for kw in remove_keywords)
]
after = len(filtered)

print(f"제거 전: {before}개 → 제거 후: {after}개 (제거: {before-after}개)")

with open("contract_risk_augmented_v2.json", "w", encoding="utf-8") as f:
    json.dump(filtered, f, ensure_ascii=False, indent=2)

from google.colab import files
files.download("contract_risk_augmented_v2.json")
print("저장 완료")

제거 전: 202개 → 제거 후: 196개 (제거: 6개)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

저장 완료


In [ ]:
from google.colab import files
files.download(OUTPUT_PATH)